# Results Analysis: Comparing Barren-Plateau Mitigation Strategies

Loads the per-seed `metrics.json` files produced by the three runners and
summarises test accuracy, test loss, training time, and the training
diagnostic (mean parameter-gradient variance) across approaches and depths.

In [ ]:
import os
import sys
from pathlib import Path

# Run from the project root regardless of the notebook's directory.
ROOT = Path.cwd()
while not (ROOT / "configs").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))

os.environ.setdefault("TF_USE_LEGACY_KERAS", "1")


## 1. Load Results

In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt

APPROACHES = ["baseline", "layerwise", "local_cost"]
DEPTHS = [4, 6, 8]

rows = []
for approach in APPROACHES:
    for depth in DEPTHS:
        base = ROOT / "results" / approach / f"depth_{depth}"
        for seed_dir in sorted(base.glob("seed_*")):
            mf = seed_dir / "metrics.json"
            if not mf.exists():
                continue
            m = json.loads(mf.read_text())
            rows.append({
                "approach": approach,
                "depth": depth,
                "seed_index": m.get("seed_index"),
                "test_acc": m["test_acc"],
                "test_loss": m["test_loss"],
                "training_time_seconds": m["training_time_seconds"],
                "n_parameters": m.get("n_parameters"),
                "grad_var": m.get("training_diagnostic", {}).get(
                    "mean_param_grad_variance"
                ),
            })
print(f"Loaded {len(rows)} completed runs.")


## 2. Test Accuracy by Approach and Depth

In [ ]:
def summarize(rows, key):
    out = {}
    for r in rows:
        out.setdefault((r["approach"], r["depth"]), []).append(r[key])
    return {k: (np.mean(v), np.std(v), len(v)) for k, v in out.items()}

acc = summarize(rows, "test_acc")
fig, ax = plt.subplots(figsize=(10, 6))
width = 0.25
for i, approach in enumerate(APPROACHES):
    means = [acc.get((approach, d), (0, 0, 0))[0] for d in DEPTHS]
    sds = [acc.get((approach, d), (0, 0, 0))[1] for d in DEPTHS]
    xs = np.arange(len(DEPTHS)) + i * width
    ax.bar(xs, means, width, label=approach, yerr=sds, capsize=3)
ax.set_xticks(np.arange(len(DEPTHS)) + width)
ax.set_xticklabels([f"L={d}" for d in DEPTHS])
ax.set_ylabel("Test accuracy")
ax.set_ylim(0, 1)
ax.legend()
ax.grid(True, axis="y", alpha=0.3)
plt.show()


## 3. Training Time

In [ ]:
tt = summarize(rows, "training_time_seconds")
fig, ax = plt.subplots(figsize=(10, 6))
for i, approach in enumerate(APPROACHES):
    means = [tt.get((approach, d), (0, 0, 0))[0] for d in DEPTHS]
    xs = np.arange(len(DEPTHS)) + i * width
    ax.bar(xs, means, width, label=approach)
ax.set_xticks(np.arange(len(DEPTHS)) + width)
ax.set_xticklabels([f"L={d}" for d in DEPTHS])
ax.set_ylabel("Training time (s)")
ax.legend()
ax.grid(True, axis="y", alpha=0.3)
plt.show()


## 4. Training Diagnostic (Mean Parameter-Gradient Variance)

In [ ]:
gv = summarize(rows, "grad_var")
fig, ax = plt.subplots(figsize=(10, 6))
for i, approach in enumerate(APPROACHES):
    means = [gv.get((approach, d), (0, 0, 0))[0] for d in DEPTHS]
    xs = np.arange(len(DEPTHS)) + i * width
    ax.bar(xs, means, width, label=approach)
ax.set_xticks(np.arange(len(DEPTHS)) + width)
ax.set_xticklabels([f"L={d}" for d in DEPTHS])
ax.set_yscale("log")
ax.set_ylabel("Mean parameter-gradient variance")
ax.legend()
ax.grid(True, axis="y", alpha=0.3)
plt.show()


## 5. Paired Comparison Report

If `run_comparison.py` has been run, prints
its `comparison.json` summary.

In [ ]:
from pathlib import Path

comp_file = ROOT / "results" / "comparison" / "comparison.json"
if comp_file.exists():
    report = json.loads(comp_file.read_text())
    for depth, pairs in report.get("pairwise_per_depth", {}).items():
        print(f"Depth {depth}:")
        for p in pairs:
            print(f"  {p['approach_a']} vs {p['approach_b']}: "
                  f"mean_diff={p['mean_diff']:+.4f}, cohens_d={p['cohens_d']:.3f}, "
                  f"ttest_p={p['ttest_p']:.3e}, holm_reject={p['holm_reject']}")
else:
    print("No comparison.json found. Run run_comparison.py after the runs complete.")


## 6. Conclusion

- Which approach achieves the best accuracy per depth.
- How training time and the gradient diagnostic vary across approaches.